In [ ]:
# Setup: imports and required NLTK resources
import os
import re
import unicodedata

import networkx as nx
import nltk
import numpy as np
import pandas as pd
import pymupdf
import spacy
from IPython.display import display
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK datasets quietly if missing
for pkg in [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "vader_lexicon",
]:
    nltk.download(pkg, quiet=True)

print("Setup complete.")


In [ ]:
# Step 1: Extract text from the sample PDF (fallback to sample text if missing)
pdf_path = os.path.join("data", "raw", "educational_content_nlp_test_document.pdf")

if os.path.exists(pdf_path):
    doc = pymupdf.open(pdf_path)
    page_count = doc.page_count
    raw_text = "\n\n".join((doc.load_page(i).get_text("text") or "") for i in range(page_count)).strip()
    doc.close()
    source_label = f"PDF ({pdf_path})"
else:
    page_count = 1
    source_label = "Fallback sample educational text"
    raw_text = (
        "Natural Language Processing is defined as a branch of artificial intelligence that helps computers "
        "understand, interpret, and generate human language. Alan Turing proposed early foundations for "
        "machine intelligence in 1950. Modern systems built with Python and spaCy process textbooks, research "
        "papers, and lecture notes automatically.\n\n"
        "Text preprocessing prepares raw documents for mathematical modeling. Tokenization splits text into "
        "sentences and words. Stopword removal removes common filler words like 'the' and 'is', while negation "
        "words such as 'not' and 'never' are preserved so meaning does not flip. Lemmatization refers to "
        "reducing inflected words to their dictionary base form using part-of-speech tags.\n\n"
        "TF-IDF stands for Term Frequency-Inverse Document Frequency and assigns higher weights to important "
        "domain terms across a document. Extractive summarization selects key original sentences directly from "
        "the source material without inventing facts. TextRank is known as a graph-based ranking algorithm "
        "that builds a sentence similarity graph and runs PageRank to score central sentences.\n\n"
        "Topic modeling discovers hidden themes inside educational documents. Latent Dirichlet Allocation is a "
        "probabilistic topic model that groups co-occurring words from a Bag-of-Words matrix into distinct "
        "topics. Named Entity Recognition identifies people, organizations, and dates in educational material."
    )

print(f"Source      : {source_label}")
print(f"Pages       : {page_count}")
print(f"Characters  : {len(raw_text)}")
print(f"Words       : {len(raw_text.split())}")
print("\nPreview (first 400 chars):\n", raw_text[:400])


In [ ]:
# Step 2: Split the extracted text into clean sentences
sentences = [re.sub(r"\s+", " ", s).strip() for s in sent_tokenize(raw_text) if s.strip()]

print(f"Total sentences: {len(sentences)}")
display(pd.DataFrame({"Sentence #": range(1, min(6, len(sentences) + 1)), "Sentence": sentences[:5]}))


In [ ]:
# Step 3: Split raw text into word tokens
word_tokens = word_tokenize(raw_text)

print(f"Total word tokens: {len(word_tokens)}")
print("First 30 tokens  :", word_tokens[:30])


In [ ]:
# Step 4: Normalize unicode, remove control chars, lowercase, and collapse spaces
cleaned_text = unicodedata.normalize("NFC", raw_text)
cleaned_text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", cleaned_text)
cleaned_text = cleaned_text.lower()
cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

print("Original preview:", raw_text[:200])
print("Cleaned preview :", cleaned_text[:200])


In [ ]:
# Step 5: Remove stopwords and punctuation while keeping negation words
protected_negations = {"no", "not", "never"}
stop_words = set(stopwords.words("english")) - protected_negations

cleaned_tokens = word_tokenize(cleaned_text)
removed_stopwords = []
filtered_tokens = []

for tok in cleaned_tokens:
    if tok in stop_words:
        removed_stopwords.append(tok)
    elif any(ch.isalnum() for ch in tok):
        filtered_tokens.append(tok)

print(f"Stopwords removed: {len(removed_stopwords)}")
print(f"Remaining tokens : {len(filtered_tokens)}")
print("Sample filtered  :", filtered_tokens[:25])


In [ ]:
# Step 6: POS-aware lemmatization back to base dictionary words
lemmatizer = WordNetLemmatizer()
pos_tags = nltk.pos_tag(filtered_tokens)

def to_wordnet_pos(tag):
    if tag.startswith("J"):
        return "a"
    if tag.startswith("V"):
        return "v"
    if tag.startswith("R"):
        return "r"
    return "n"

lemmatized_tokens = [lemmatizer.lemmatize(tok, pos=to_wordnet_pos(tag)) for tok, tag in pos_tags]

lemma_df = pd.DataFrame([
    {"Original": tok, "POS": tag, "Lemma": lem}
    for (tok, tag), lem in zip(pos_tags[:15], lemmatized_tokens[:15])
])
display(lemma_df)


In [ ]:
# Step 7: Build TF-IDF vectors over preprocessed sentences and rank top terms
preprocessed_sentences = []
for sent in sentences:
    toks = [t.lower() for t in word_tokenize(sent) if any(c.isalnum() for c in t) and t.lower() not in stop_words]
    lemmas = [lemmatizer.lemmatize(t, pos=to_wordnet_pos(tag)) for t, tag in nltk.pos_tag(toks)]
    preprocessed_sentences.append(" ".join(lemmas))

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(preprocessed_sentences)
feature_names = tfidf_vectorizer.get_feature_names_out()

# Rank vocabulary by average TF-IDF weight across sentences
mean_weights = np.asarray(tfidf_matrix.mean(axis=0)).ravel()
top_indices = np.argsort(mean_weights)[::-1][:10]

print(f"Vocabulary size: {len(feature_names)}")
print(f"Matrix shape   : {tfidf_matrix.shape}")
display(pd.DataFrame([
    {"Term": feature_names[i], "Mean TF-IDF": round(float(mean_weights[i]), 4)}
    for i in top_indices
]))


In [ ]:
# Step 8: Extractive summarization using TextRank (TF-IDF + Cosine Similarity + PageRank)
seen_keys = set()
unique_sentences = []
for s in sentences:
    key = re.sub(r"\s+", " ", s.strip().lower())
    if key not in seen_keys:
        seen_keys.add(key)
        unique_sentences.append(s)

n_sent = len(unique_sentences)
num_summary_sentences = min(5, n_sent)

# Vectorize unique sentences and compute pairwise similarity
sent_vectors = TfidfVectorizer(sublinear_tf=True).fit_transform(unique_sentences)
sim_matrix = cosine_similarity(sent_vectors)
np.fill_diagonal(sim_matrix, 0.0)

# Build weighted similarity graph and run PageRank
graph = nx.Graph()
graph.add_nodes_from(range(n_sent))
for i in range(n_sent):
    for j in range(i + 1, n_sent):
        w = float(sim_matrix[i, j])
        if w > 0.0:
            graph.add_edge(i, j, weight=w)

if graph.number_of_edges() > 0:
    scores = nx.pagerank(graph, alpha=0.85, weight="weight")
else:
    scores = {i: 1.0 / max(1, n_sent) for i in range(n_sent)}

# Pick top-scoring sentences and restore original document order
ranked_idx = sorted(range(n_sent), key=lambda i: scores[i], reverse=True)
selected_idx = sorted(ranked_idx[:num_summary_sentences])
extractive_summary = " ".join(unique_sentences[i] for i in selected_idx)
compression_ratio = len(extractive_summary.split()) / max(1, len(raw_text.split()))

print(f"Graph nodes: {graph.number_of_nodes()}, Graph edges: {graph.number_of_edges()}")
print(f"Compression ratio: {compression_ratio:.2%}\n")
print("Extractive Summary:\n", extractive_summary)

display(pd.DataFrame([
    {
        "Sentence #": i + 1,
        "PageRank Score": round(scores[i], 4),
        "Selected": i in selected_idx,
        "Sentence": unique_sentences[i],
    }
    for i in ranked_idx[:8]
]))


In [ ]:
# Step 9: Information extraction (spaCy NER, noun-chunk keyphrases, regex definitions)
nlp = spacy.load("en_core_web_sm")
doc = nlp(raw_text)

# Extract deduplicated named entities
entities, seen_ents = [], set()
for ent in doc.ents:
    key = (ent.text.strip().lower(), ent.label_)
    if key not in seen_ents:
        seen_ents.add(key)
        entities.append({"Entity": ent.text.strip(), "Label": ent.label_})

# Extract top noun-chunk keyphrases ranked by frequency + multi-word boost
generic_words = {"this", "that", "these", "those", "it", "they", "we", "you", "the", "a", "an"}
phrase_counts = {}
for chunk in doc.noun_chunks:
    phrase = " ".join(chunk.text.split())
    if chunk.root.pos_ not in {"NOUN", "PROPN"} or phrase.lower() in generic_words:
        continue
    if any(w.lower() not in generic_words for w in phrase.split()):
        phrase_counts[phrase] = phrase_counts.get(phrase, 0) + 1

keyphrases = sorted(
    [
        {"Keyphrase": p, "Score": round(c + 0.1 * (len(p.split()) - 1), 2)}
        for p, c in phrase_counts.items()
    ],
    key=lambda x: (-x["Score"], x["Keyphrase"]),
)[:10]

# Extract rule-based concept definitions from sentences
def_patterns = [
    (r"^(.+?)\s+(?:is|are|can\s+be)\s+defined\s+as\s+(.+)$", "is defined as"),
    (r"^(.+?)\s+refers\s+to\s+(.+)$", "refers to"),
    (r"^(.+?)\s+means\s+(.+)$", "means"),
    (r"^(.+?)\s+(?:is|are)\s+known\s+as\s+(.+)$", "is known as"),
    (r"^(.+?)\s+is\s+(?:a|an|the)\s+(.+)$", "is a/an/the"),
]

definitions, seen_concepts = [], set()
for sent in sentences:
    clean_s = sent.rstrip(".!? ")
    if len(clean_s) > 500:
        continue
    for pattern, label in def_patterns:
        m = re.match(pattern, clean_s, flags=re.IGNORECASE)
        if not m:
            continue
        concept = re.sub(r"^(?:the|a|an)\s+", "", m.group(1).strip(" ,;:-"), flags=re.IGNORECASE)
        definition = m.group(2).strip()
        max_words = 5 if label == "is a/an/the" else 7
        if concept and definition and len(concept.split()) <= max_words and concept.lower() not in seen_concepts:
            seen_concepts.add(concept.lower())
            definitions.append({"Concept": concept, "Pattern": label, "Definition": definition})
            break

print(f"Entities: {len(entities)} | Keyphrases shown: {len(keyphrases)} | Definitions: {len(definitions)}")
display(pd.DataFrame(entities[:15]))
display(pd.DataFrame(keyphrases))
display(pd.DataFrame(definitions[:10]))


In [ ]:
# Step 10: Topic modelling with Latent Dirichlet Allocation (LDA)
chunk_size = 5
chunks = [" ".join(sentences[i : i + chunk_size]) for i in range(0, len(sentences), chunk_size)]

bow_vectorizer = CountVectorizer(stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]{2,}\b", min_df=1)
bow_matrix = bow_vectorizer.fit_transform(chunks)
bow_terms = bow_vectorizer.get_feature_names_out()

n_topics = min(3, max(1, len(bow_terms)))
lda = LatentDirichletAllocation(n_components=n_topics, max_iter=20, random_state=42, learning_method="batch")
lda.fit(bow_matrix)

# Collect top 8 keywords per topic
topic_rows = []
for topic_idx, weights in enumerate(lda.components_):
    probs = weights / max(1e-12, weights.sum())
    top_idx = np.argsort(probs)[::-1][:8]
    topic_rows.append({
        "Topic": f"Topic {topic_idx}",
        "Top Keywords": ", ".join(f"{bow_terms[i]} ({probs[i]:.3f})" for i in top_idx),
    })

# Compute dominant topic for each chunk
doc_topic_dist = lda.transform(bow_matrix)
chunk_rows = []
for i, chunk in enumerate(chunks):
    dist = doc_topic_dist[i]
    dom = int(np.argmax(dist))
    chunk_rows.append({
        "Chunk #": i + 1,
        "Dominant Topic": f"Topic {dom} ({dist[dom]:.2f})",
        "Snippet": chunk[:90] + ("..." if len(chunk) > 90 else ""),
    })

print(f"Chunks: {len(chunks)} | BoW shape: {bow_matrix.shape} | Perplexity: {lda.perplexity(bow_matrix):.2f}")
display(pd.DataFrame(topic_rows))
display(pd.DataFrame(chunk_rows))


In [ ]:
# Step 11: Student feedback sentiment analysis using VADER
student_feedbacks = [
    "The explanations and diagrams made TextRank and TF-IDF super clear!",
    "Too much theory without enough examples; I got confused during topic modeling.",
    "The lecture slides cover preprocessing steps.",
    "Great breakdown of classical NLP versus neural approaches, really helpful for exams!",
    "Audio quality in the recorded session was poor and hard to follow.",
]

vader = SentimentIntensityAnalyzer()
sentiment_rows = []

for fb in student_feedbacks:
    scores = vader.polarity_scores(fb)
    compound = scores["compound"]
    if compound >= 0.05:
        label = "Positive"
    elif compound <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"
    sentiment_rows.append({
        "Feedback": fb,
        "Pos": scores["pos"],
        "Neu": scores["neu"],
        "Neg": scores["neg"],
        "Compound": compound,
        "Sentiment": label,
    })

sentiment_df = pd.DataFrame(sentiment_rows)
display(sentiment_df)
print("\nSentiment counts:\n", sentiment_df["Sentiment"].value_counts())
